# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset name:', metadata.name)
print('Description:', metadata.description)

# Optional: display other metadata
print('\nAuthors:', getattr(metadata, 'author', None))
print('Cite as:', getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', None)))
print('Version:', getattr(metadata, 'version', None))
print('Keywords:', getattr(metadata, 'keywords', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

To interact with the dataset, list available record sets and their associated field (column) `@id`s.

In [ ]:
# List available record sets in the dataset
print('Available record sets:')
record_sets = []
try:
    # Some Croissant datasets expose record_sets as metadata.record_sets (plural)
    record_sets = getattr(metadata, 'record_sets', getattr(metadata, 'recordSet', []))
except Exception as e:
    print(e)

# If it's empty or not present, attempt to derive from metadata fields
if not record_sets or isinstance(record_sets, dict):
    # Try to access from 'dataset.record_sets' property
    try:
        record_sets = dataset.record_sets
    except AttributeError:
        record_sets = []

if isinstance(record_sets, dict):
    # If it's a dict (should be a list), treat as single record set
    record_sets = [record_sets]

if record_sets:
    for rs in record_sets:
        rec_id = getattr(rs, '@id', getattr(rs, 'id', None)) if hasattr(rs, '@id') else (rs.get('@id') if isinstance(rs, dict) else rs)
        rec_name = getattr(rs, 'name', None) if hasattr(rs, 'name') else rs.get('name') if isinstance(rs, dict) else rs
        print(f' - {rec_id}: {rec_name}')
        try:
            fields = getattr(rs, 'fields', getattr(rs, 'field', []))
        except Exception:
            fields = []
        if fields:
            print('    Fields:')
            for f in fields:
                field_id = getattr(f, '@id', f.get('@id') if isinstance(f, dict) else f)
                field_name = getattr(f, 'name', f.get('name') if isinstance(f, dict) else f)
                print(f'     - {field_id}: {field_name}')
else:
    # Fallback for datasets with only one default record set or CSV
    print("No explicit record sets found. Try exploring records directly.")

# Try to preview the first 2 records in the main record set
print('\nSample record contents:')
try:
    for idx, record in enumerate(dataset.records()):
        print(record)
        if idx >= 1:
            break
except Exception as e:
    print("Failed to iterate records:", e)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Try to extract all record sets or the default one if only one exists.
from collections import OrderedDict

dataframes = {}
record_set_ids = []

# 1. Try explicit record sets (most croissant datasets use a default ID if none is present)
try:
    all_record_sets = getattr(metadata, 'recordSet', getattr(metadata, 'record_sets', []))
    # If it's dict, wrap as list
    if isinstance(all_record_sets, dict):
        all_record_sets = [all_record_sets]
    # Get ids if possible
    for rs in all_record_sets:
        if isinstance(rs, dict):
            rec_id = rs.get('@id', None)
        elif hasattr(rs, '@id'):
            rec_id = getattr(rs, '@id')
        else:
            rec_id = rs
        if rec_id:
            record_set_ids.append(rec_id)
except Exception as e:
    pass

# 2. If none found, use default (mlcroissant assumes default if not provided)
if not record_set_ids:
    record_set_ids = [None]  # None lets mlcroissant pick the default

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if isinstance(record_set_id, str) or record_set_id is None:
        key = record_set_id if record_set_id is not None else 'default_record_set'
        dataframes[key] = pd.DataFrame(records)
    else:
        key = str(record_set_id)
        dataframes[key] = pd.DataFrame(records)

# Select the proper key for main data table
main_set_id = record_set_ids[0] if isinstance(record_set_ids[0], str) or record_set_ids[0] is None else str(record_set_ids[0])
main_df = dataframes[main_set_id]
print('Available columns:', main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll select one numeric field (e.g., `MW [kDa]`, molecular weight in kilodaltons if available), filter records, normalize, and group by another attribute (e.g., `Description`). All fields should be referenced by their exact column heading in the DataFrame, corresponding to their `@id` in the Croissant schema if known.

In [ ]:
# Print candidate numeric fields
print('Numeric-like fields:')
numeric_candidates = [col for col in main_df.columns if main_df[col].dtype in ['float64', 'int64'] or any(x in col.lower() for x in ['cover', 'count', 'mw', 'num', 'quantity', 'abun'])]
print(numeric_candidates)

# Choose a likely numeric field name, e.g. 'MW [kDa]' or adapt as needed from actual column names listed above
if len(numeric_candidates) > 0:
    numeric_field_id = numeric_candidates[0]  # First found; adjust index for others
else:
    numeric_field_id = main_df.select_dtypes(include='number').columns[0] if not main_df.select_dtypes(include='number').empty else None

if numeric_field_id is not None:
    # Convert to numeric, handle errors just in case
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    # Apply a basic threshold filter (10 is arbitrary, adapt this for real exploration)
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a possible categorical field (e.g. 'Description', 'Sample', etc.)
    # Pick one likely group-by field:
    candidate_group_fields = [col for col in main_df.columns if col != numeric_field_id and main_df[col].dtype == 'object']
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing means):")
        print(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the selected numeric field, and if grouping is possible, plot the mean per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if 'filtered_df' in locals() and group_field:
        # Plot group means
        group_means = filtered_df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)[:20]
        plt.figure(figsize=(10,6))
        sns.barplot(x=group_means.values, y=group_means.index, orient='h')
        plt.title(f'Mean {numeric_field_id} by {group_field} (top 20)')
        plt.xlabel(f'Mean {numeric_field_id}')
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print('No suitable numeric field for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset on extracellular vesicle mass spectrometry analysis contains comprehensive protein feature records.
- Using `mlcroissant`, we loaded metadata, explored record structure, extracted tabular data, and performed basic filtering, normalization, and grouping.
- Simple visualizations give a first look into the numeric field distributions and class-wise means, suitable for downstream analysis or machine learning.

Further analyses can involve outlier removal, imputation of missing values, and advanced modeling based on biological hypotheses.